# ERSP Analysis — REAL TIME (non-time-warped), GO-locked
Raw signals are loaded from network storage across three patient groups using a single format-agnostic loader that handles TRC, EDF, and H5 files (load_first_raw_in_dir). Non-neural channels are removed (filter_aux_channels), and for EL patients only electrodes with anatomical labels are kept. Signals are then rereferenced to the average of white matter contacts defined per patient (apply_wm_reref), and line noise is removed at each harmonic only if a real peak is detected, with notch strength set automatically (notch_mains_harmonics). Trial timing comes from photodiode triggers saved as TSV files per patient, and trials are kept only if the stimulus lasted at least 0.5s and the response no more than 10s, with IQR used to remove remaining outliers (collect_trials). ERSPs are computed using short-time Fourier transform and warped so each trial is split 50/50 between stimulus and post-stimulus, baseline corrected before stimulus onset (compute_ersp). White matter channels are skipped. Outputs per channel are an ERSP plot, a high-gamma heatmap sorted by trial duration (plot_hg_trials), and for clustering a raw matrix and a clean image. QC outputs are two PSDs (before and after processing) and a full recording montage with trial markers (plot_montage_overview).

For each patient, loads and preprocesses raw neural signals, then runs one or both of two parallel pipelines controlled by boolean flags.

## Processing Steps (shared for all patients)

#### 1. Data Loading
- Builds patient-specific paths (raw + prep directories)
- Loads raw signals using format-agnostic loader (TRC/EDF/H5)
- Removes auxiliary channels (ECG, DC, markers etc.)

#### 2. Channel Filtering (EL patients only)
- **SEEG patients**: keeps only channels with `_` in name (e.g. `A_L6`)
- **Grid patients** (e.g. EL044): keeps only channels matching defined prefixes with a digit (e.g. `Pa1`, `T17`, `postP3`)
- Skips patient entirely if no neural channels remain

#### 3. Preprocessing
- Saves **PSD before processing** (if flag on)
- Applies **white matter rereferencing**
- Applies **adaptive mains notch filtering**
- Saves **PSD after processing** (if flag on)

#### 4. Trial Collection
- Reads trial TSV files from `prep0`
- Applies hard duration filters: `min_stim_s=0.5`, `max_post_s=10`
- Trims outliers using IQR method
- Saves QC report and histogram


## Pipeline A — ERSP Pipeline
*Runs if `RUN_ERSP_PIPELINE=True`*

- Saves **montage overview plot** with trial onset/offset markers
- For each condition and channel:
  - Computes **ERSP** (time-frequency power map)
  - Saves **HG trial plot** (`.png`, GO-locked, sorted by response duration)
  - Saves **HG trials plot** (high-gamma, trial-by-trial heatmap)

## Pipeline B — Cluster Export
*Runs if `RUN_CLUSTER_EXPORT=True`*

- Skips non-neural, bad, and WM channels
- For each condition and channel:
  - Reuses ERSP result if Pipeline A also ran (no recomputation)
  - Saves **ERSP matrix** (`.npy`) for clustering input
  - Saves **clean ERSP image** (`.png`) for clustering input

## Outputs
| Product | Location | Pipeline |
|---|---|---|
| HG plots (**GO-locked**, sorted by response duration) | `outputs/05_ERSP_LM_RAWONLY_RealTime/<pid>/LM/HG/<cond>` | A |
| ERSP matrix (**real time**, GO-locked) | `outputs/05_ERSP_LM_RAWONLY_RealTime/<pid>/LM/ERSP_matrix/<cond>` | B |
| ERSP clean PNG | `outputs/05_ERSP_LM_RAWONLY_RealTime/<pid>/LM/ERSP_clean/<cond>` | B |
| Report + montage | `outputs/05_ERSP_LM_RAWONLY_RealTime/<pid>/LM/Report` | A |

**Everything lands in one root.** Per-channel ERSP TIFFs and the PSD overviews are
140's job and are not repeated here — they are what makes `04_ersp_LM` 264 GB.
Files are tagged `_RT_GO`, so they can never be confused with 140's `_TN` cubes.

**Not time-warped.** 140 stretches every trial onto 300 bins (0% = stimulus onset,
50% = GO). The response follows GO after a variable natural delay, and warping
spends exactly that delay. Here epochs are cut in real seconds and centred on the
GO cue (`sample_offsets`). There is **no speech-onset event in the data**, so t=0
is GO, not the moment the patient spoke.

## Imports & run controls 
(the only place you change things is here and cell 4)


In [ ]:
# ============================================================
# 150_ERSP_analysis_pipeline_noTwarping.ipynb
# Cell 1 — Imports & run controls  ·  REAL-TIME (non-time-warped), GO-locked
# ============================================================
import os, glob, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from itertools import groupby
import csv

from functions import lf_io_utils as io, lf_trials as tr, lf_ersp as fe, config as cfg
from functions.config import (PAT_PATIENTS, PAT_PRESETS, EL_PATIENTS, EL_PRESETS,
                               MICROEPI_MAT_PATIENTS, MICROEPI_MAT_PRESETS, COND_ALIAS)
import LFfunctions_PDextract as LF
from LF_pd import load_patient_raw

# ------------------------------------------------------------------
# RUN CONTROLS  ← edit these before each run
# ------------------------------------------------------------------
BLOCK = "LM"

# Toggle sections
RUN_PD_EXTRACTION    = False   # trial TSVs already extracted by 140; reuse them
RUN_ERSP_PIPELINE    = True
RUN_CLUSTER_EXPORT   = True
DO_MONTAGE_PSD_PLOTS = False    # PSD/montage QC belongs to 140; not repeated here

# ------------------------------------------------------------------
# REAL-TIME (non-time-warped) SETTINGS  — the only difference from 140
# ------------------------------------------------------------------
# 140 writes TIME-NORMALISED cubes: every trial is stretched onto 300 bins with
# 0% = stimulus onset, 50% = GO. That makes trials commensurable, but the
# response follows GO after a variable delay, and warping spends that delay --
# the quantity a response-timing analysis is trying to measure.
#
# Here nothing is warped. Epochs are cut in REAL SECONDS and centred on the GO
# cue, which in these TSVs is the stimulus OFFSET (`sample_offsets`). Note there
# is no speech-onset event anywhere in the data, so t=0 is GO, not the moment
# the patient actually spoke.
RT_MODE   = "RT"
RT_ALIGN  = "go"
# Generous on purpose. Stimulus durations reach ~1.5 s, so -2.5 s keeps the whole
# stimulus visible before GO, and RT_MAX_POST_S below caps the response to match.
# Short windows that start near the event let the spectrogram's
# edge frames inflate into a spurious peak -- measured: a (0.0, 2.0) window moved
# the recovered latency of a synthetic GO-locked burst from +0.25 s to +0.54 s,
# while (-2.5, 4.0) and (-1.0, 3.0) both recovered +0.25 s.
RT_WINDOW = (-2.5, 5.0)

# Responses longer than 5 s are invalid trials. This has to match RT_WINDOW's
# post-GO reach: cfg.max_post_s is 10.0, so without this a 7 s trial would be
# KEPT by collect_trials and then silently truncated by the epoch window --
# a trial that is half-measured rather than either included or rejected.
# cfg.max_post_s itself is untouched, so 140 keeps its 10 s.
RT_MAX_POST_S = 5.0

# The CLEAN PNG is what MOBA shows when you click an electrode, so it has to be
# readable. Two fixes, both to the PICTURE only -- the .npy keeps every bin:
#   * crop above 400 Hz. The cube spans 0-500 Hz in 129 bins; the top of that range
#     carries nothing and only costs frame height.
#   * draw it wide. A square frame squashed a 129 x ~480 cube into its frequency
#     extent, turning a 1.5 s response into a narrow vertical blob.
RT_CLEAN_SHOW_HZ = 400.0
RT_CLEAN_ROWS    = int(round(RT_CLEAN_SHOW_HZ / 500.0 * 128)) + 1   # 52 of 129
RT_CLEAN_FIGSIZE = (7.5, 3.0)

# ONE output root. Everything else in the project is assumed time-normalised.
RT_SCRIPT_NAME = "05_ERSP_LM_RAWONLY_RealTime"
run_root_ersp  = os.path.join(cfg.outputs_root, RT_SCRIPT_NAME)
run_root_raw   = run_root_ersp          # HG plots and cubes share one tree

# ERSP params (from config.py)
# BASELINE, stated rather than inherited. compute_ersp's RT branch reads
# `baseline_calc_w`, NOT `baseline_w`. Passing only baseline_w (as 140 does) left
# the ERSPParams dataclass default (-0.4,-0.1) in force, so cfg.baseline_w was
# passed but dead and cfg.baseline_calc_w never arrived at all -- and the HG plot
# beside it used (-0.6,-0.1). Two baselines in one figure pair.
# Both now come from cfg.baseline_w, so the cube and the HG plot agree.
# NOTE this makes the RT cubes differ from 140's TN cubes, which still use the
# (-0.4,-0.1) default. Deliberate: changing 140 would invalidate the shipped
# 04_ersp_LM_RAWONLY tree, and the two are not comparable anyway (warped vs not).
RT_BASELINE = cfg.baseline_w          # (-0.6, -0.1) s relative to STIMULUS onset

ersp_params = fe.ERSPParams(
    nperseg=cfg.nperseg, nfft=cfg.nfft, noverlap=cfg.noverlap,
    baseline_w=RT_BASELINE, baseline_calc_w=RT_BASELINE,
    proportions=cfg.proportions,
    n_time_bins=cfg.n_time_bins, vmin=cfg.vmin, vmax=cfg.vmax, fmax=cfg.fmax
)

print(f"Controls loaded — REAL TIME, mode={RT_MODE} align={RT_ALIGN} "
      f"window={RT_WINDOW}s")
print(f"  -> {run_root_ersp}")

In [ ]:
# ============================================================
# Cell 2 — Helper functions
# (implementations live in lf_ersp.py and lf_io_utils.py)
# ============================================================

# Direct aliases
notch_mains_harmonics = fe.notch_mains_harmonics
fill_nans_nearest     = fe.fill_nans_nearest
save_clean_png        = fe.save_clean_png
plot_psd_overview     = fe.plot_psd_overview
_is_non_neural        = io._is_non_neural
_ensure               = io.ensure_dir

# MicroEPI .mat helpers (used for G-01..G-06 — saved as preset['pat_name'] e.g. PAT_6704)
import functions.lf_micromacro as mm

# Thin cfg-binding wrappers (keep pipeline cells unchanged)
def apply_notch_with_audit(signals, fs, patient_id, pid_raw):
    return fe.apply_notch_with_audit(
        signals, fs, patient_id, pid_raw,
        notch_patients=getattr(cfg, "notch_patients", []),
        mains_base=getattr(cfg, "mains_base", 50.0),
        fmax=getattr(cfg, "fmax", 500.0),
        repeats=getattr(cfg, "notch_repeats", 1),
        peak_z_thresh=getattr(cfg, "notch_peak_z_thresh", 3.0),
    )

def apply_wm_reref(signals, names, patient_id, *, electrodes_tsv_pattern=None):
    """Returns (signals, reref_label, wm_skip_set).

    `electrodes_tsv_pattern` overrides the auto-resolved BIDS path — used for
    MicroEPI .mat patients whose TSV lives outside the standard cohort layout.

    Raises ValueError if cfg.reref_type is not 'WM', if no WM channels are
    found for the patient, or if WM rereferencing ultimately failed to apply.
    All outputs in this pipeline MUST be WM rereferenced.
    """
    if str(cfg.reref_type).upper() != "WM":
        raise ValueError(
            f"[reref] cfg.reref_type is '{cfg.reref_type}' — only 'WM' is "
            f"allowed in this pipeline. Update config.py and re-run."
        )
    wm_idx = io.wm_indices_for_patient(patient_id, names,
                                        electrodes_tsv_pattern=electrodes_tsv_pattern)
    if not wm_idx:
        raise ValueError(
            f"[reref] {patient_id}: no WM channels found — cannot apply WM "
            f"rereferencing. Check the electrodes TSV and WM threshold."
        )
    bad = getattr(cfg, "bad_channels_manual", {}).get(patient_id, [])
    signals_r, used, excluded = fe.apply_wm_reference_with_exclusions(
        signals, names, wm_idx, bad)
    if not used:
        raise ValueError(
            f"[reref] {patient_id}: WM channels were found but none were used "
            f"after exclusions — cannot guarantee WM rereferencing. "
            f"Check bad_channels_manual and available WM channels."
        )
    return signals_r, "WM", set(used) | set(excluded)


def _load_signals_and_prep_for_patient(pid_raw):
    """Route a patient to its loader.

    For MicroEPI .mat patients (G-01..G-06): load the combined macros +
    micros signal matrix via `mm.load_and_concatenate_mats` +
    `mm.build_combined_signals`. Returns the BIDS electrodes.tsv from
    `MICROEPI_MAT_PRESETS` plus the `is_micro` mask so cell 9 can apply the
    macro-only WM reref + optional anchor reref for micros.

    Everything else (PAT, EL) falls through to the standard TRC/EDF/H5
    loader and returns `is_micro=None`.

    Returns
    -------
    patient_id              : str (preset['pat_name'] for MicroEPI .mat patients)
    signals, names, fs      : as returned by the chosen loader
    prep_dir                : where collect_trials should read from
    electrodes_tsv_pattern  : explicit pattern for WM reref (or None)
    is_micro                : (n_channels,) bool mask for MicroEPI patients;
                              None for PAT / EL.
    """
    pid_str = str(pid_raw)
    if pid_str in getattr(cfg, "MICROEPI_MAT_PATIENTS", []):
        preset = cfg.MICROEPI_MAT_PRESETS[pid_str]
        patient_id = preset["pat_name"]
        d = mm.load_and_concatenate_mats(preset["data_dir"], preset["mat_files"])
        signals, names, is_micro = mm.build_combined_signals(
            d["data_ecog"], d["data_micro"], d["chans_ecog"], d["chans_micro"])
        fs = d["fs"]
        prep_dir = os.path.join(os.path.dirname(preset["data_dir"]), "prep0")
        electrodes_tsv = preset["electrodes_tsv"]
        print(f"  [paths] data_dir: {preset['data_dir']}")
        print(f"  [paths] prep_dir: {prep_dir}")
        print(f"  [microepi] {signals.shape[1]} channels = "
              f"{int((~is_micro).sum())} macros + {int(is_micro.sum())} micros")
    else:
        patient_id, raw_dir, prep_dir = io.build_paths_for_patient(pid_raw, cfg.block_name)
        signals, names, fs = io.load_first_raw_in_dir(raw_dir)
        electrodes_tsv = None  # auto-resolve from cohort
        is_micro = None
    return patient_id, signals, names, fs, prep_dir, electrodes_tsv, is_micro


print("Helpers loaded.")

## Part 1 — Photodiode / trial extraction
Runs `LF_pd.load_patient_raw` and `LFfunctions_PDextract` for each patient in `PD_PATIENTS`.
Output: TSV timing files saved to each patient's `prep0` folder.
Set `RUN_PD_EXTRACTION = False` in Cell 1 to skip.

### PD extraction loop (PAT / EL / MicroEPI unified, calls LF_pd as-is)


In [ ]:
import glob
for pid in ["EL046"]:#["EL030","EL034","EL035","EL036","EL037","EL038","EL040","EL042","EL043","EL044","EL045"]:
    new_dir = rf"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_BERN\{pid}\task_FBM\data_LM\raw"
    has_h5  = bool(glob.glob(f"{new_dir}\\*_LM.h5"))
    has_edf = bool(glob.glob(f"{new_dir}\\*_LM.edf"))
    has_tsv = bool(glob.glob(f"{new_dir}\\*.tsv"))
    print(f"  {pid}: h5={has_h5} edf={has_edf} tsv={has_tsv}  ({new_dir})")

In [ ]:
import importlib; importlib.reload(cfg)
from functions import lf_io_utils as io, lf_trials as tr, lf_ersp as fe, config as cfg
from functions.config import (PAT_PATIENTS, PAT_PRESETS, EL_PATIENTS, EL_PRESETS,
                               MICROEPI_MAT_PATIENTS, MICROEPI_MAT_PRESETS, COND_ALIAS)
from LF_pd import load_patient_raw

if not RUN_PD_EXTRACTION:
    print("[skip] PD extraction (RUN_PD_EXTRACTION=False)")
else:
    # Active patients per group (empty list = skip that group)
    PAT_PATIENTS          = [] #6953
    EL_PATIENTS           = ["EL048"]  # e.g. [,"EL048","EL042","EL043","EL044","EL045"]
    MICROEPI_MAT_PATIENTS = []  # e.g. ["G-01","G-02","G-03","G-04","G-05","G-06"]

    # ────────────────────────────────────────────────────────────────────
    # PAT + EL loop — TRC / EDF / H5 loaders + standard PD detection
    # ────────────────────────────────────────────────────────────────────
    all_patients = (
        [(pid, "PAT", PAT_PRESETS) for pid in PAT_PATIENTS] +
        [(pid, "EL",  EL_PRESETS)  for pid in EL_PATIENTS]
    )

    for pid, group, presets in all_patients:
        preset = presets.get(pid)
        if preset is None:
            print(f"[skip] {pid}: no preset"); continue

        patient_id = f"PAT_{pid}" if group == "PAT" else pid
        print(f"\n=== {patient_id} ===")

        try:
            if group == "PAT":
                base_path = fr"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_HUG\{patient_id}\task_FBM\data_{BLOCK}\raw"
                raw_signals, channel_names, sampling_rate = io.load_trc_and_signals(glob.glob(os.path.join(base_path, "*.TRC"))[0])
                save_path = os.path.join(os.path.dirname(base_path), "prep0")

            elif group == "EL":
                info          = load_patient_raw(pid, block_name=BLOCK, use_el_mat_fallback=False, verbose=True)
                raw_signals   = info["raw_signals"]
                sampling_rate = info["sampling_rate"]
                channel_names = list(info["channel_names"])
                print(channel_names)
                save_path     = info["save_path"]
                exp_file      = info["matching_files_onsets"][0] if info["matching_files_onsets"] else None

            if group != "EL":
                exp_files = glob.glob(os.path.join(base_path, "*.tsv")) or glob.glob(os.path.join(base_path, "*.txt"))
                exp_file  = exp_files[0] if exp_files else None

            lower_map = {str(c).lower(): str(c) for c in channel_names}
            trig_key  = preset["trig"].lower()
            if trig_key not in lower_map:
                print(f"  [warn] trigger '{preset['trig']}' not found — skipping"); continue
            pd_name = lower_map[trig_key]

            mt = preset["manual_trig"]
            manual_path = None
            if mt:
                manual_path = mt if os.path.isabs(mt) else os.path.join(save_path, mt)
                if not os.path.exists(manual_path):
                    print(f"  [warn] manual triggers not found: {manual_path}")
                    manual_path = None

            print("FLIPS!!! ", preset["flip"])
            on_abs, off_abs, metrics = LF.get_trigger_indexes_photodiode(
                raw_signals=raw_signals, sampling_rate=sampling_rate,
                channel_names=channel_names, trig_name=pd_name,
                time_range=preset["time_range"], threshold_val=0.40,
                flip_trigs=preset["flip"],
                trial_ids=preset["trial_ids"], invalid_trials=preset["invalid_trials"],
                ignore_invalid=False, fake_trials=preset["fake_trials"],
                extra_table_path=exp_file, manual_trigs_path=manual_path,
                return_extra_metrics=True,
            )
            print(f"  Paired trials: {len(on_abs)}")
            LF.parse_and_save(pid, patient_id, on_abs, off_abs, metrics,
                              sampling_rate, save_path, BLOCK, exp_file,
                              preset["trial_ids"], preset["trig"],
                              cond_alias=COND_ALIAS)

        except Exception as e:
            print(f"[error] {patient_id}: {e}")

    # ────────────────────────────────────────────────────────────────────
    # MicroEPI .mat PD extraction (G-01..G-06, Geneva cohort)
    # Mirrors notebook 11_'s flow exactly:
    #   1) load + concat per-block .mat exports via mm.load_and_concatenate_mats
    #   2) photodiode detection on the dedicated PD trace
    #   3) merge condition_name / resp_accuracy / trial_idx from the
    #      behavioral events TSV
    #   4) write per-condition prep0 TSVs in the same format collect_trials
    #      reads downstream (so cell 9 doesn't care this is MicroEPI)
    # ────────────────────────────────────────────────────────────────────
    for pid in MICROEPI_MAT_PATIENTS:
        preset = cfg.MICROEPI_MAT_PRESETS.get(pid)
        if preset is None:
            print(f"\n[skip] MicroEPI-{pid}: no preset in cfg.MICROEPI_MAT_PRESETS"); continue
        patient_id = preset["pat_name"]
        print(f"\n=== MicroEPI-{pid} → {patient_id} ===")
        try:
            # 1) Load + concatenate per-block .mat files
            d = mm.load_and_concatenate_mats(preset["data_dir"], preset["mat_files"])
            fs = d["fs"]
            print(f"  signals: ecog {d['data_ecog'].shape}  micro {d['data_micro'].shape}  fs={fs}")

            # 2) Photodiode event detection
            beh_path = os.path.join(preset["data_dir"], preset["tsv_file"])
            on_abs, off_abs = mm.extract_events_from_photodiode(
                d["photodiode"], fs,
                time_range=preset.get("time_range", (0, -1)),
                trial_ids=preset.get("trial_ids", []) or None,
                invalid_trials=preset.get("invalid_trials", []) or None,
                fake_trials=preset.get("fake_trials", []) or None,
                extra_table_path=beh_path,
                do_plot=False,
            )
            print(f"  photodiode: {len(on_abs)} onsets, {len(off_abs)} offsets")

            # 3) Pull condition_name / resp_accuracy / trial_idx from the behavioral TSV
            beh = LF._read_trial_table(beh_path)
            dfl = beh["raw_df"].rename(columns=str.lower)
            def _pick(cols):
                return next((dfl[c].astype(str).to_numpy() for c in cols if c in dfl), None)
            condition_name = _pick(["category", "blockname"])
            resp_accuracy  = _pick(["response_type", "responseaccuracy"])
            trial_idx_col  = _pick(["exemplar", "stimnumber"])
            trial_ids_for_save = (preset.get("trial_ids") or
                                  ([str(x).lower() for x in condition_name]
                                   if condition_name is not None else []))

            # 4) Write per-condition prep0 TSVs (same format as PAT/EL)
            prep_dir = os.path.join(os.path.dirname(preset["data_dir"]), "prep0")
            LF.save_onsets_offsets_by_condition(
                patient_id=patient_id, block_name=BLOCK,
                onsets=on_abs, offsets=off_abs, sampling_rate=fs,
                trial_ids=trial_ids_for_save, out_dir=prep_dir,
                condition_name=condition_name, resp_accuracy=resp_accuracy,
                trial_idx=trial_idx_col,
                cond_alias=COND_ALIAS,
                trigger_label=preset.get("trig", "photodiode"),
            )
            print(f"  [{patient_id}] PD extraction done -> {prep_dir}")
        except Exception as e:
            print(f"[error] MicroEPI {pid}: {e}")

    print("\n[PD extraction done]")

## Part 2 — ERSP pipeline + cluster export
Reads raw data and `prep0` TSVs. Toggles:
- `RUN_ERSP_PIPELINE = True` → GO-locked HG plots, Report, montage QC into `05_ERSP_LM_RAWONLY_RealTime/`
- `RUN_CLUSTER_EXPORT = True` → per-channel `ERSP_matrix/*_RT_GO.npy` and `ERSP_clean/*.png` into the same root (consumed by `05_FBM_ResponseTiming`, **not** by `02_FBM_Clustering`, which expects 300 warped bins)

Both flags can be on simultaneously — the loop computes the ERSP once per (channel, condition) and dispatches outputs to whichever pipeline is enabled.

In [ ]:
# cfg.patient_ids=["EL034","EL037", "EL038", "EL040", "EL045","EL044","EL030","EL033","EL048"]
# cfg.patient_ids=["EL044","EL033","EL048"] 
# cfg.patient_ids=[2868, 3066, 3390, 3415, 3455, 3965, 3975, 3780]

In [ ]:
# Cohort for the real-time run: 6 MicroEPI + 14 Bern + 10 HUG = 30.
cfg.patient_ids = [#"G-06", "G-04", "G-05", "G-01", "G-02", "G-03",
                   #"EL030", "EL033", "EL034", "EL035", "EL036", "EL037", "EL038",
                   "EL040", "EL042", "EL043", "EL044", "EL045", "EL046", "EL048",
                   "PAT_3455", "PAT_2868", "PAT_3066", "PAT_3301", "PAT_3390",
                   "PAT_3415", "PAT_3965", "PAT_3975", "PAT_3780", "PAT_6953"]
#assert len(cfg.patient_ids) == len(set(cfg.patient_ids)) == 30
print(f"cohort: {len(cfg.patient_ids)} patients")
# PAT_6953 is the one patient with raw but no extracted trials -- it will be
# skipped with "no trials" while RUN_PD_EXTRACTION is False.
# cfg.patient_ids=["G-03"]


In [ ]:
import gc, os
import pandas as pd

# ──────────────────────────────────────────────────────────────────────────
# Per-cell toggles
# ──────────────────────────────────────────────────────────────────────────
# When False, MicroEPI micros are skipped in the per-channel ERSP/HG/cluster
# loop — only macros are processed (matches the pre-refactor behaviour).
# When True (default), micros are treated like any other channel: ERSP/HG
# plots get saved and the cluster export writes one .npy per micro.
INCLUDE_MICROEPI_MICROS = True

if not RUN_ERSP_PIPELINE and not RUN_CLUSTER_EXPORT:
    print("[skip] both pipelines disabled")
    # Stubs so the per-patient cells below fail with an explanation rather than
    # a bare NameError.
    def run_patients(ids):
        print("[skip] set RUN_ERSP_PIPELINE / RUN_CLUSTER_EXPORT and re-run this cell")
    def wm_report():
        print("[skip] nothing was run")
else:
    # Per-patient summary collected during the loop and printed/saved at the end.
    # Status codes: ok | ok-no-trials | error-load | error-reref | error-other
    wm_report_rows = []

    def process_patient(pid_raw):
        """
        Process one patient end-to-end. Wrapped in a function so all heavy
        intermediates (raw signals, ERSP cubes, matplotlib figures) become
        garbage-collectable on return — keeps RAM bounded across 16+ patients
        without needing a kernel restart.
        """
        report = {
            "pid_raw": str(pid_raw),
            "patient_id": "",
            "status": "",
            "n_channels_in": 0,
            "n_channels_neural": 0,
            "n_channels_unknown_dropped": 0,
            "n_channels_used": 0,
            "n_wm_used": 0,
            "wm_channels_used": "",
            "wm_channels_excluded_as_bad": "",
            "error": "",
        }
        try:
            patient_id, signals, names, fs, prep_dir, _wm_tsv, is_micro = \
                _load_signals_and_prep_for_patient(pid_raw)
            report["patient_id"] = patient_id
            report["n_channels_in"] = len(names)
            is_microepi = is_micro is not None

            # ── Aux drop (ECG / DC / markers) — applies to all cohorts.
            # For MicroEPI the loader already returns curated macros + micros,
            # so this is usually a no-op but kept for symmetry / safety.
            if is_microepi:
                # filter_aux_channels returns new arrays — keep is_micro aligned
                _n_before = len(names)
                kept_idx = [i for i, nm in enumerate(names) if not _is_non_neural(nm)]
                signals  = signals[:, kept_idx]
                names    = [names[i] for i in kept_idx]
                is_micro = is_micro[kept_idx]
                if len(names) < _n_before:
                    print(f"  [{patient_id}] aux drop: {_n_before} → {len(names)} channels")
            else:
                signals, names, *_ = io.filter_aux_channels(signals, names)

            # ── EL prefix / grid filter — EL ONLY (no MicroEPI).
            if (not is_microepi) and str(pid_raw).startswith("EL"):
                if pid_raw in cfg.EL_GRID_PATIENTS:
                    prefixes = cfg.EL_GRID_KEEP_PREFIXES.get(pid_raw, ())
                    keep = [i for i, nm in enumerate(names)
                            if any(str(nm).startswith(p) for p in prefixes)
                            and any(c.isdigit() for c in str(nm))]
                else:
                    keep = [i for i, nm in enumerate(names) if ("_" in str(nm) or "-" in str(nm))]
                signals = signals[:, keep]
                names   = [names[i] for i in keep]
                if len(names) == 0:
                    print(f"[skip] {patient_id}: no neural channels")
                    report["status"] = "no-neural-channels"
                    return report
            report["n_channels_neural"] = len(names)

            # ── EXPERIMENT-WINDOW CROP (EL ONLY) ─────────────────────────
            # MicroEPI doesn't crop here — PD detection already used the
            # preset's time_range, and the .mat exports are already trimmed
            # to the recording session.
            crop_offset = 0
            _preset = cfg.EL_PRESETS.get(pid_raw) if (not is_microepi and str(pid_raw).startswith("EL")) else None
            if _preset and _preset.get("time_range") and _preset["time_range"][1] > 0:
                t0, t1 = _preset["time_range"]
                s0 = max(0, int(t0 * fs))
                s1 = min(signals.shape[0], int(t1 * fs))
                if s1 > s0 and (s1 - s0) < signals.shape[0]:
                    full_s = signals.shape[0] / fs
                    signals     = np.ascontiguousarray(signals[s0:s1, :])
                    crop_offset = s0
                    print(f"  [{patient_id}] cropped to {t0:.0f}s..{t1:.0f}s "
                          f"({(s1-s0)/fs:.1f}s of {full_s:.1f}s)")

            # ── PER-PATIENT CHANNEL-NAME OVERRIDE (EL ONLY) ──────────────
            if (not is_microepi) and patient_id in getattr(cfg, "STRIP_HEMI_PATIENTS", set()):
                import re as _re
                _hemi_re = _re.compile(r"_(?:[LR])(?=\d)")
                names = [_hemi_re.sub("", str(nm)) for nm in names]
                print(f"  [{patient_id}] stripped _L#/_R# per cfg.STRIP_HEMI_PATIENTS  "
                      f"-> first few: {names[:8]}")

            # ── DROP "UNKNOWN" PARCELLATION CHANNELS (EL/PAT ONLY) ───────
            # Skipped for MicroEPI: micros aren't in BIDS, and macros are
            # already explicitly enumerated in the .mat exports we trust.
            if not is_microepi:
                try:
                    unk_idx = set(io.unknown_indices_for_patient(
                        patient_id, names, electrodes_tsv_pattern=_wm_tsv,
                    ))
                except Exception as _e_unk:
                    print(f"[warn] {patient_id}: Unknown-channel lookup failed ({_e_unk}); keeping all")
                    unk_idx = set()
                if patient_id in getattr(cfg, "MIXED_GRID_DEPTH_PATIENTS", set()):
                    keep_prefixes = cfg.MIXED_GRID_KEEP_PREFIXES.get(patient_id, ())
                    if keep_prefixes:
                        protected = {i for i in unk_idx
                                     if any(str(names[i]).startswith(p) for p in keep_prefixes)}
                        if protected:
                            print(f"  [{patient_id}] protected {len(protected)} grid channels "
                                  f"from Unknown drop (prefixes={keep_prefixes})")
                        unk_idx -= protected
                if unk_idx:
                    report["n_channels_unknown_dropped"] = len(unk_idx)
                    keep = [i for i in range(len(names)) if i not in unk_idx]
                    signals = signals[:, keep]
                    names   = [names[i] for i in keep]
                    print(f"  [{patient_id}] dropped {len(unk_idx)} 'Unknown' channels (no parcellation in TSV)")

            # ── REREFERENCING ────────────────────────────────────────────
            # MicroEPI:  WM reref on macros only via mm.apply_wm_reref_selective.
            #            Micros optionally re-referenced to a single anchor
            #            electrode (preset['micro_reref_anchor']); otherwise
            #            left raw.
            # EL grid (cfg.EL_GRID_PATIENTS) without WM: reref='NONE' fallback.
            # Everything else: standard apply_wm_reref via lf_ersp.
            if is_microepi:
                pid_str = str(pid_raw)
                preset  = cfg.MICROEPI_MAT_PRESETS.get(pid_str, {})
                wm_names_raw = mm.derive_wm_channels_from_electrodes_tsv(_wm_tsv)
                print(f"  [{patient_id}] WM channels from TSV: {len(wm_names_raw)} "
                      f"-> {wm_names_raw[:6]}{'...' if len(wm_names_raw) > 6 else ''}")
                try:
                    signals, wm_used, wm_excl = mm.apply_wm_reref_selective(
                        signals, names, wm_names_raw, is_micro,
                        apply_wm_to_micros=False)
                except Exception as _e_reref:
                    print(f"[error] {patient_id}: macro WM reref failed — {_e_reref}")
                    report["status"] = "error-reref"
                    report["error"]  = str(_e_reref)
                    return report
                reref   = "WM"
                wm_skip = set()  # macros that ARE WM stay in the loop; selective reref doesn't zero them
                report["n_wm_used"]                   = len(wm_used)
                report["wm_channels_used"]            = "|".join(sorted(wm_used))
                report["wm_channels_excluded_as_bad"] = ""

                # Optional anchor reref for micros
                anchor_name = preset.get("micro_reref_anchor")
                signals, anchor_used = mm.apply_micro_anchor_reref(
                    signals, names, is_micro, anchor_name)
                if anchor_used:
                    print(f"  [{patient_id}] micros re-referenced to anchor: {anchor_used}")
                else:
                    print(f"  [{patient_id}] micros left raw (no micro_reref_anchor in preset)")
            else:
                is_grid    = str(pid_raw) in getattr(cfg, "EL_GRID_PATIENTS", set())
                wm_all     = io.wm_labels_for_patient(patient_id, electrodes_tsv_pattern=_wm_tsv)
                _bad_norm_for_report = {io.normalize_label(b)
                                        for b in getattr(cfg, "bad_channels_manual", {}).get(patient_id, [])}
                _name_norm = [io.normalize_label(n) for n in names]
                wm_in_sig  = [n for n in wm_all if n in _name_norm]
                wm_usable  = [n for n in wm_in_sig if n not in _bad_norm_for_report]

                if wm_usable:
                    try:
                        signals, reref, wm_skip = apply_wm_reref(
                            signals, names, patient_id, electrodes_tsv_pattern=_wm_tsv,
                        )
                        wm_excluded_norm = [n for n in wm_in_sig if n in _bad_norm_for_report]
                        report["n_wm_used"]                    = len(wm_usable)
                        report["wm_channels_used"]             = "|".join(sorted(wm_usable))
                        report["wm_channels_excluded_as_bad"]  = "|".join(sorted(wm_excluded_norm))
                    except ValueError as _e_reref:
                        print(f"[error] {patient_id}: WM reref failed — {_e_reref}")
                        report["status"] = "error-reref"
                        report["error"]  = str(_e_reref)
                        return report
                elif is_grid:
                    # Grid patient without WM contacts. If listed in
                    # cfg.GRID_CAR_PATIENTS, CAR each array separately;
                    # otherwise fall back to no rereferencing.
                    if str(pid_raw) in getattr(cfg, "GRID_CAR_PATIENTS", set()):
                        _bad_car = getattr(cfg, "bad_channels_manual", {}).get(patient_id, [])
                        _car_prefixes = cfg.EL_GRID_KEEP_PREFIXES.get(pid_raw, None)
                        signals, car_groups = fe.apply_grid_car(
                            signals, names,
                            group_prefixes=_car_prefixes,
                            bad_channels=_bad_car, min_group=2)
                        reref   = "CAR"
                        wm_skip = set()
                        _summary = ", ".join(f"{k}({len(v)})" for k, v in sorted(car_groups.items()))
                        print(f"[note] {patient_id}: per-grid CAR applied — groups: {_summary}")
                        report["n_wm_used"]                   = 0
                        report["wm_channels_used"]            = "CAR:" + _summary
                        report["wm_channels_excluded_as_bad"] = "|".join(sorted(_bad_car))
                        # not a failure — overwritten to 'ok' at end if pipeline completes
                    else:
                        print(f"[note] {patient_id}: grid patient with no WM contacts — "
                              f"falling back to reref='NONE' (no rereferencing applied)")
                        reref   = "NONE"
                        wm_skip = set()
                        report["n_wm_used"]                   = 0
                        report["wm_channels_used"]            = ""
                        report["wm_channels_excluded_as_bad"] = ""
                        report["status"]                      = "ok-no-wm-grid"
                else:
                    msg = f"no WM channels available for {patient_id} (not a grid patient)"
                    print(f"[error] {msg}")
                    report["status"] = "error-reref"
                    report["error"]  = msg
                    return report

            signals = apply_notch_with_audit(signals, fs, patient_id, pid_raw)

            if RUN_ERSP_PIPELINE and DO_MONTAGE_PSD_PLOTS:
                fe.plot_psd_overview(
                    signals=signals, fs=fs, names=names,
                    save_root=io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "PSD_clean"),
                    patient_id=patient_id, block_name=cfg.block_name,
                    fmax=cfg.fmax, mains_base=getattr(cfg, "mains_base", 50.0), dpi=600,
                )

            report_dir  = io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "Report")
            report_path = os.path.join(report_dir, f"{patient_id}_IQR.tsv")
            cond_groups = tr.collect_trials(prep_dir, fs, outlier_method="IQR",
                                            iqr_k=cfg.iqr_k, report_path=report_path,
                                            patient_id=patient_id, max_post_s=RT_MAX_POST_S,
                                            condition_aliases=cfg.COND_ALIAS)
            if not cond_groups:
                print(f"[skip] {patient_id}: no trials")
                report["status"] = "ok-no-trials"
                return report

            # Re-base trigger sample indices into the cropped signal.
            # Only EL patients are cropped (crop_offset > 0); MicroEPI / PAT
            # never enter this branch.
            if crop_offset > 0:
                rebased = {}
                n_samples = signals.shape[0]
                for cond, (on, off, tend) in cond_groups.items():
                    on   = np.asarray(on)   - crop_offset
                    off  = np.asarray(off)  - crop_offset
                    tend = np.asarray(tend) - crop_offset
                    ok = (on >= 0) & (tend <= n_samples)
                    if (~ok).any():
                        print(f"  [{patient_id} | {cond}] dropped {(~ok).sum()} "
                              f"trials outside crop window")
                    rebased[cond] = (on[ok], off[ok], tend[ok])
                cond_groups = rebased

            if RUN_ERSP_PIPELINE:
                tr.plot_montage_overview(
                    signals=signals, fs=fs, names=names,
                    cond_groups=cond_groups, save_dir=report_dir, patient_id=patient_id,
                    fmt="png",     # was TIFF: 9 files, ~1 GB each, 73% of the tree
                )
                hg_root   = io.patient_output_dir(run_root_ersp, patient_id, cfg.block_name, "HG")

            if RUN_CLUSTER_EXPORT:
                mat_root = _ensure(io.patient_output_dir(run_root_raw, patient_id, cfg.block_name, "ERSP_matrix"))
                img_root = _ensure(io.patient_output_dir(run_root_raw, patient_id, cfg.block_name, "ERSP_clean"))
                _bad_norm = {io.normalize_label(b)
                             for b in getattr(cfg, "bad_channels_manual", {}).get(patient_id, [])}
                skip      = set(n for n in names if _is_non_neural(n))
                skip     |= {n for n in names if io.normalize_label(n) in _bad_norm}
                skip     |= wm_skip

            # Build the set of micro channel names (used to optionally skip
            # micros in the per-channel loop when INCLUDE_MICROEPI_MICROS is
            # False). For non-MicroEPI patients this is always empty.
            micro_names_set = set()
            if is_microepi and not INCLUDE_MICROEPI_MICROS:
                micro_names_set = {names[i] for i in range(len(names)) if is_micro[i]}

            for cond, (onsets, offsets, trial_ends) in cond_groups.items():
                if RUN_ERSP_PIPELINE:
                    hg_dir   = _ensure(os.path.join(hg_root, cond))
                if RUN_CLUSTER_EXPORT:
                    out_mat  = _ensure(os.path.join(mat_root, cond))
                    out_png  = _ensure(os.path.join(img_root, cond))

                print(f"  {cond}: ", end="", flush=True)
                for ci, chan_name in enumerate(names):
                    if chan_name in wm_skip: continue
                    if chan_name in micro_names_set: continue
                    if RUN_CLUSTER_EXPORT and chan_name in skip and not RUN_ERSP_PIPELINE: continue

                    res = fe.compute_ersp(
                        signals=signals, fs=fs, onsets=onsets, offsets=offsets, channel_idx=ci,
                        trial_ends=trial_ends, mode=RT_MODE, time_window=RT_WINDOW,
                        align=RT_ALIGN, params=ersp_params,
                    )

                    if RUN_ERSP_PIPELINE:
                        # No per-channel ERSP TIFF here. That is 140's job and it is
                        # what makes 04_ersp_LM 264 GB; this tree carries only the
                        # HG trial plots and the cubes.
                        fe.plot_hg_trials(
                            signals=signals, fs=fs, onsets=onsets, offsets=offsets, channel_idx=ci,
                            chan_name=chan_name, patient_id=patient_id, condition=cond, reref_type=reref,
                            time_window=RT_WINDOW, baseline_w=RT_BASELINE,
                            hg_band=cfg.hg_band, smooth_ms=cfg.hg_smooth_ms,
                            vmin=cfg.hg_vmin, vmax=cfg.hg_vmax,
                            save_dir=hg_dir, add_separators=False, sort_ascending=True,
                            trial_end_indices=trial_ends,
                            # t=0 is GO; rows ordered by how long the response lasted,
                            # so the green trial-end ticks form a staircase.
                            sort_by="resp", align="go", fmt="png",
                        )

                    if RUN_CLUSTER_EXPORT and chan_name not in skip:
                        A = np.array(res["avg_db"], float)
                        if np.isnan(A).any():
                            print(f"[warn] {patient_id} {cond} {chan_name}: {int(np.isnan(A).sum())} NaNs → filling")
                            fill_nans_nearest(A)
                        # 140 tags TN and leaves RT untagged, which would make these
                        # indistinguishable from legacy untagged files. Tag both.
                        _m = str(res["meta"]["mode"]).upper()
                        mode_tag = "_TN" if _m == "TN" else (
                            "_RT_GO" if res["meta"].get("align") == "go" else "_RT")
                        stem = f"{patient_id}_{cond}_{reref}_ERSP_{chan_name}{mode_tag}"
                        np.save(os.path.join(out_mat, f"{stem}.npy"), A)
                        save_clean_png(A, vmin=ersp_params.vmin, vmax=ersp_params.vmax,
                                       path_png=os.path.join(out_png, f"{stem}_CLEAN.png"),
                                       keep_rows=RT_CLEAN_ROWS, figsize=RT_CLEAN_FIGSIZE)
                        del A
                    del res
                print(" done")

            # Preserve the "ok-no-wm-grid" tag if we set it earlier;
            # otherwise this is a fully-normal "ok".
            if not report["status"]:
                report["status"] = "ok"
            return report
        except Exception as e:
            print(f"[error] {pid_raw}: {e}")
            report["status"] = report["status"] or "error-other"
            report["error"]  = str(e)
            return report

    # ────────────────────────────────────────────────────────────────────────
    # Main loop. Per-patient bodies run inside process_patient(); we then
    # explicitly close all matplotlib figures + GC to keep memory bounded.
    # ────────────────────────────────────────────────────────────────────────

    # ────────────────────────────────────────────────────────────────────────
    # Per-patient driver. One patient per cell below, so any single patient can
    # be re-run on its own without touching the others. Re-running a patient
    # REPLACES its row in the report rather than appending a second one.
    # ────────────────────────────────────────────────────────────────────────
    def _row_pid(row):
        for k in row:
            if "patient" in k.lower():
                return str(row[k])
        return None

    def run_patients(ids):
        """Process one patient (or a few). Safe to call repeatedly."""
        if isinstance(ids, str):
            ids = [ids]
        last = None
        for pid_raw in ids:
            print("\n"); print(pid_raw)
            row = process_patient(pid_raw)
            pid = _row_pid(row)
            if pid is not None:
                wm_report_rows[:] = [r for r in wm_report_rows if _row_pid(r) != pid]
            wm_report_rows.append(row)
            last = row
            plt.close("all"); gc.collect()
        return last

    def wm_report():
        """Print and save the report over whatever has been run so far."""
        df_report = pd.DataFrame(wm_report_rows)
        print("\n" + "=" * 72)
        print("WM REREFERENCING REPORT (per patient)")
        print("=" * 72)
        if len(df_report) == 0:
            print("(no patients processed)")
        else:
            # Compact print: status | n_wm_used | n_channels_used | error?
            for _, r in df_report.iterrows():
                status_tag = {
                    "ok":                  "  OK   ",
                    "ok-no-trials":        "OK-NOTR",
                    "ok-no-wm-grid":       "OK-GRID",
                    "no-neural-channels":  "NO-CHAN",
                    "error-reref":         "REREF✗ ",
                    "error-load":          "LOAD✗  ",
                    "error-other":         "ERR✗   ",
                    "":                    "??     ",
                }.get(r["status"], r["status"])
                line = (f"  {status_tag}  {r['patient_id']:<14}  "
                        f"in={r['n_channels_in']:>4}  "
                        f"neural={r['n_channels_neural']:>4}  "
                        f"unknown_dropped={r['n_channels_unknown_dropped']:>3}  "
                        f"used={r['n_channels_used']:>4}  "
                        f"wm={r['n_wm_used']:>3}")
                if r["error"]:
                    line += f"  err='{r['error'][:60]}'"
                print(line)
            n_ok   = (df_report["status"] == "ok").sum()
            n_fail = (df_report["status"].astype(str).str.startswith("error")).sum()
            print("-" * 72)
            print(f"Summary: {n_ok}/{len(df_report)} patients fully processed   "
                  f"{n_fail} failed   "
                  f"(total Unknown dropped: {df_report['n_channels_unknown_dropped'].sum()};   "
                  f"total WM used: {df_report['n_wm_used'].sum()})")

            # Persist (TSV; survives kernel restart and is reviewable in Excel/etc.)
            try:
                out_tsv = os.path.join(run_root_raw, "wm_reref_report.tsv")
                os.makedirs(os.path.dirname(out_tsv) or ".", exist_ok=True)
                df_report.to_csv(out_tsv, sep="\t", index=False)
                print(f"  saved -> {out_tsv}")
            except Exception as _e_save:
                print(f"  [warn] could not save report TSV: {_e_save}")


## Per-patient runs

One cell per patient so any single patient can be re-run on its own. Run the definition cell above once first. Re-running a patient replaces its row in the report; `wm_report()` at the bottom summarises whatever has been run.


### G-06 — MicroEPI (Geneva) · French · files written as `PAT_6854`

**macro + micro** · 160 trials in 3 block(s) · 0 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (FR) (53), reading (53)


In [ ]:
run_patients("G-06")


### G-04 — MicroEPI (Geneva) · English/French · files written as `PAT_6704`

**macro + micro** · 160 trials in 4 block(s) · 1 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), reading (53), audio (FR) (52), audio (EN) (1)

> Only patient with an **English** trial mixed into the French auditory block (`auditory_naming_eng`) — now normalised to `audio`.


In [ ]:
run_patients("G-04")


### G-05 — MicroEPI (Geneva) · French · files written as `PAT_6684`

**macro + micro** · 157 trials in 3 block(s) · 0 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), reading (53), audio (FR) (50)


In [ ]:
run_patients("G-05")


### G-01 — MicroEPI (Geneva) · French · files written as `PAT_5515`

**macro + micro** · 160 trials in 3 block(s) · 0 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (FR) (53), reading (53)


In [ ]:
run_patients("G-01")


### G-02 — MicroEPI (Geneva) · French · files written as `PAT_5533`

**macro + micro** · 160 trials in 3 block(s) · 0 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (FR) (53), reading (53)


In [ ]:
run_patients("G-02")


### G-03 — MicroEPI (Geneva) · French · files written as `PAT_6619`

**macro + micro** · 160 trials in 3 block(s) · 0 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (FR) (53), reading (53)

> Failed on memory in the first pass; runs alone.


In [ ]:
run_patients("G-03")


### EL030 — Bern · German

**depth electrodes (SEEG)** · 151 trials in 3 block(s) · 1 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (DE) (53), reading (44)


In [ ]:
run_patients("EL030")


### EL033 — Bern · German

**depth electrodes (SEEG)** · 159 trials in 3 block(s) · 1 manual bad channel(s) · 3 unique trial file(s)

Blocks: audio (DE) (53), picture (53), reading (53)


In [ ]:
run_patients("EL033")


### EL034 — Bern · German

**depth electrodes (SEEG)** · 159 trials in 3 block(s) · 5 manual bad channel(s) · 3 unique trial file(s)

Blocks: audio (DE) (53), reading (53), picture (53)


In [ ]:
run_patients("EL034")


### EL035 — Bern · German

**depth electrodes (SEEG)** · 160 trials in 3 block(s) · 13 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (DE) (53), reading (53)


In [ ]:
run_patients("EL035")


### EL036 — Bern · German

**depth electrodes (SEEG)** · 160 trials in 3 block(s) · 9 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (DE) (53), reading (53)


In [ ]:
run_patients("EL036")


### EL037 — Bern · German

**depth electrodes (SEEG)** · 160 trials in 3 block(s) · 39 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (DE) (53), reading (53)

> Most manual bad channels in the cohort (39).


In [ ]:
run_patients("EL037")


### EL038 — Bern · German

**depth electrodes (SEEG)** · 141 trials in 3 block(s) · 5 manual bad channel(s) · 3 unique trial file(s)

Blocks: reading (51), picture (51), audio (DE) (39)


In [ ]:
run_patients("EL038")


### EL040 — Bern · German

**depth electrodes (SEEG)** · 160 trials in 3 block(s) · 3 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (DE) (53), reading (53)


In [ ]:
run_patients("EL040")


### EL042 — Bern · German

**depth electrodes (SEEG)** · 160 trials in 3 block(s) · 1 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (DE) (53), reading (53)


In [ ]:
run_patients("EL042")


### EL043 — Bern · German

**strip hemisphere override** · 106 trials in 2 block(s) · 1 manual bad channel(s) · 2 unique trial file(s)

Blocks: audio (DE) (53), reading (53)

> Reading block missing. Strip-hemisphere override applies.


In [ ]:
run_patients("EL043")


### EL044 — Bern · German

**grid (ECoG) · per-grid CAR · grid + depth** · 53 trials in 1 block(s) · 13 manual bad channel(s) · 1 unique trial file(s)

Blocks: audio (DE) (53)

> Grid patient — WM reref falls back to per-grid CAR. Only the auditory block ran, and 8 contacts are still unlocalised.


In [ ]:
run_patients("EL044")


### EL045 — Bern · German

**depth electrodes (SEEG)** · 160 trials in 3 block(s) · 1 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), audio (DE) (53), reading (53)


In [ ]:
run_patients("EL045")


### EL046 — Bern · unlabelled

**depth electrodes (SEEG)** · 53 trials in 1 block(s) · 1 manual bad channel(s) · 1 unique trial file(s)

Blocks: picture (53)

> Only one block, and the condition label carries no language suffix.


In [ ]:
run_patients("EL046")


### EL048 — Bern · unlabelled

**depth electrodes (SEEG)** · 158 trials in 1 block(s) · 1 manual bad channel(s) · 3 unique trial file(s)

Blocks: nan (158)

> Condition label carries no language suffix.


In [ ]:
run_patients("EL048")


### PAT_3455 — HUG (Geneva) · unlabelled

**depth electrodes (SEEG)** · 152 trials in 3 block(s) · 1 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (53), reading (51), auditory_naming (48)


In [ ]:
run_patients("PAT_3455")


### PAT_2868 — HUG (Geneva) · unlabelled

**depth electrodes (SEEG)** · 160 trials in 3 block(s) · 1 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), auditory_naming (53), reading (53)


In [ ]:
run_patients("PAT_2868")


### PAT_3066 — HUG (Geneva) · unlabelled

**depth electrodes (SEEG)** · 154 trials in 3 block(s) · 1 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), auditory_naming (50), reading (50)


In [ ]:
run_patients("PAT_3066")


### PAT_3301 — HUG (Geneva) · unlabelled

**depth electrodes (SEEG)** · 43 trials in 1 block(s) · 1 manual bad channel(s) · 1 unique trial file(s)

Blocks: picture (43)

> Picture block only.


In [ ]:
run_patients("PAT_3301")


### PAT_3390 — HUG (Geneva) · unlabelled

**depth electrodes (SEEG)** · 160 trials in 3 block(s) · 1 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), auditory_naming (53), reading (53)


In [ ]:
run_patients("PAT_3390")


### PAT_3415 — HUG (Geneva) · unlabelled

**grid + depth** · 160 trials in 3 block(s) · 17 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (54), auditory_naming (53), reading (53)

> Grid + depth: surface contacts protected by prefix, WM reref still runs on depths.


In [ ]:
run_patients("PAT_3415")


### PAT_3965 — HUG (Geneva) · unlabelled

**depth electrodes (SEEG)** · 51 trials in 1 block(s) · 5 manual bad channel(s) · 1 unique trial file(s)

Blocks: reading (51)

> Picture block only.


In [ ]:
run_patients("PAT_3965")


### PAT_3975 — HUG (Geneva) · German

**depth electrodes (SEEG)** · 150 trials in 3 block(s) · 1 manual bad channel(s) · 3 unique trial file(s)

Blocks: audio (DE) (50), picture (50), reading (50)


In [ ]:
run_patients("PAT_3975")


### PAT_3780 — HUG (Geneva) · French

**depth electrodes (SEEG)** · 150 trials in 3 block(s) · 1 manual bad channel(s) · 3 unique trial file(s)

Blocks: picture (51), audio (FR) (50), reading (49)


In [ ]:
run_patients("PAT_3780")


### PAT_6953 — HUG (Geneva) · unlabelled

**depth electrodes (SEEG)** · 0 trials in 0 block(s) · 1 manual bad channel(s) · 0 unique trial file(s)

Blocks: none found

> **No extracted trials on disk** — needs PD extraction before it can run.


In [ ]:
run_patients("PAT_6953")


## Report over everything run so far


In [ ]:
wm_report()
